# TranSTR — NextQA (DINOv3 + GroundingDINO + LoRA + NCOD + HUM)

Bản port từ `train-transtr-gdino-lora-ncod-hum-integrated.ipynb` sang **NextQA**.

**Khác biệt chính so với bản causal**:
- Annotation đọc từ 3 CSV `train.csv / val.csv / test.csv` (cột: `video, question, answer, a0..a4, qid, type`).
- `qns_key = f"{video}_{qid}"`.
- Type ∈ {CW, CH, TN, TP, TC, DC, DL, DO}; family-id mapping: D*=0, T*=1, C*=2 (HUM kích hoạt khi `q_family_id >= 2`).
- Video features `.pt` được scan đệ quy (rglob) vì `extractvit.ipynb` ghi theo cây thư mục.
- Eval theo spec NextQA, dump JSON `{qid: {answer, prediction}}`.
- Mọi config train (bs, lr, epoch, LoRA, NCOD, HUM, lambdas) **giữ nguyên** để so sánh công bằng.

In [ ]:
# CELL 1: Git clone & setup (giống bản causal)
import os
REPO_URL = "https://github.com/DanielQH07/tranSTR_Casual.git"
REPO_NAME = "tranSTR_Casual"
BRANCH = "full_mode"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} -b {BRANCH}
else:
    print("Repo already exists.")

if os.path.basename(os.getcwd()) != REPO_NAME:
    try:
        target_dir = os.path.join(os.getcwd(), REPO_NAME, "causalvid")
        if os.path.exists(target_dir):
            os.chdir(target_dir)
        elif os.path.exists(REPO_NAME):
            os.chdir(REPO_NAME)
        print(f"Changed directory to: {os.getcwd()}")
    except Exception as e:
        print(f"Could not set working directory: {e}")

In [ ]:
# CELL 2: Install + W&B login
print('=== CELL 2: W&B Setup ===')
!pip install -q wandb peft --upgrade

try:
    import torchao
    from packaging import version
    torchao_ver = version.parse(getattr(torchao, '__version__', '0.0.0'))
    if torchao_ver < version.parse('0.16.0'):
        print(f'Incompatible torchao {torchao_ver}; uninstalling to avoid PEFT ImportError...')
        !pip uninstall -y torchao
except Exception:
    pass

import wandb
WANDB_API_KEY = ''
WANDB_PROJECT = 'transtr-nextqa-gdino-lora-ncod-hum'
WANDB_ENTITY = None
if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print('\u2705 W&B logged in')
else:
    print('\u26a0\ufe0f WANDB_API_KEY trống — wandb.init sẽ chạy anonymous/offline tuỳ runtime.')

In [ ]:
# CELL 3: 
import os, os.path as osp, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
from utils.util import set_seed, set_gpu_devices
from networks.model import VideoQAmodel
from DataLoaderNextQA import (NextQADataset, scan_video_pt, scan_gdino, NEXTQA_TYPE_TO_FAMILY, GDINO_DIM)
from eval_mc_nextqa import accuracy_metric as accuracy_metric_nextqa
print('Imports OK')

In [ ]:
# CELL 4: Train/eval functions (NCOD + HUM + Verifier/Knowledge) — y nguyên bản causal
print('=== CELL 4: Functions ===')

def _unpack_batch(batch):
    if len(batch) == 7:
        ff, of, q, a, ans_id, qns_key, q_family_id = batch
    elif len(batch) == 6:
        ff, of, q, a, ans_id, qns_key = batch
        q_family_id = None
    else:
        raise ValueError(f'Unexpected batch format with {len(batch)} elements')
    return ff, of, q, a, ans_id, qns_key, q_family_id

def _compute_sample_indices(qns_keys, key_to_idx, device):
    idxs = [key_to_idx.get(str(k), -1) for k in qns_keys]
    if any(i < 0 for i in idxs):
        missing = [str(qns_keys[i]) for i, v in enumerate(idxs) if v < 0][:5]
        raise KeyError(f'Missing qns_key in key_to_idx mapping: {missing}')
    return torch.tensor(idxs, dtype=torch.long, device=device)

def train_epoch_integrated(model, optimizer_model, optimizer_u, U, loader, xe, bce, device,
                           epoch, key_to_idx, accumulation_steps=4, hum_alpha=1.0,
                           lambda_verifier=0.2, lambda_knowledge=0.1):
    model.train()
    total_loss=total_l1=total_l2=total_verifier=total_knowledge=0.0
    correct, total = 0, 0
    optimizer_model.zero_grad(); optimizer_u.zero_grad()
    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
        if q_family_id is None:
            q_family_id = torch.zeros_like(tgt)
        else:
            q_family_id = q_family_id.to(device)

        sample_indices = _compute_sample_indices(qns_keys, key_to_idx, device)
        out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
        logits = out['logits']
        fused_logits = out.get('fused_score', logits)
        verifier_logits = out.get('verifier_logits', logits)
        knowledge_logits = out.get('knowledge_score', None)

        probs = torch.softmax(logits, dim=1)
        y_onehot = F.one_hot(tgt, num_classes=logits.size(-1)).float()
        u_batch = U[sample_indices].unsqueeze(1)

        ce_per_sample = -torch.sum(y_onehot * torch.log(torch.clamp(probs, min=1e-12)), dim=1)
        shifted_probs = torch.clamp(probs + (u_batch.detach() * y_onehot), min=1e-12, max=1.0)
        lum_loss = -torch.sum(y_onehot * torch.log(shifted_probs), dim=1)
        hum_loss = (1.0 + hum_alpha * u_batch.detach().squeeze(1)) * ce_per_sample
        hard_mask = (q_family_id >= 2)
        l1 = torch.where(hard_mask, hum_loss, lum_loss).mean()

        verifier_loss = bce(verifier_logits, y_onehot)
        if knowledge_logits is not None:
            knowledge_loss = bce(knowledge_logits, y_onehot)
        else:
            knowledge_loss = torch.tensor(0.0, device=device)

        model_loss = l1 + lambda_verifier * verifier_loss + lambda_knowledge * knowledge_loss
        (model_loss / accumulation_steps).backward()

        shifted_det = probs.detach() + (u_batch * y_onehot)
        l2 = F.mse_loss(shifted_det, y_onehot)
        (l2 / accumulation_steps).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_model.step(); optimizer_model.zero_grad()
            optimizer_u.step(); optimizer_u.zero_grad()
            with torch.no_grad():
                U.clamp_(0.0, 0.99)

        total_l1 += l1.item(); total_l2 += l2.item()
        total_verifier += verifier_loss.item(); total_knowledge += knowledge_loss.item()
        total_loss += (model_loss + l2).item()
        correct += (fused_logits.argmax(-1) == tgt).sum().item(); total += tgt.size(0)
        pbar.set_postfix({'loss': total_loss/(batch_idx+1), 'acc': correct/max(total,1)*100})

    n = len(loader)
    return (total_loss/n, total_l1/n, total_l2/n, total_verifier/n, total_knowledge/n,
            correct/max(total,1)*100)

def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, _, q_family_id = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            correct += (logits.argmax(-1) == tgt).sum().item(); total += tgt.size(0)
    return correct/max(total,1)*100

print('Functions defined.')

In [ ]:
# CELL 5: Paths & Config (NextQA on Kaggle, align with Train_full_mode)
print('=== CELL 5: Paths & Config ===')

# === Inputs (NextQA) ===
VIDEO_FEATURE_PATH  = '/kaggle/input/nextqa-dinov3-feat/features'        # chứa *.pt theo cây con
GDINO_FEATURE_PATH  = '/kaggle/input/nextqa-gdino-roi-merged'            # chứa {video_id}.pkl
TRAIN_CSV = '/kaggle/input/nextqa-labels/train.csv'
VAL_CSV   = '/kaggle/input/nextqa-labels/val.csv'
TEST_CSV  = '/kaggle/input/nextqa-labels/test.csv'

BASE = '/kaggle/working'
MODEL_DIR = os.path.join(BASE, 'models_nextqa')
os.makedirs(MODEL_DIR, exist_ok=True)

RUN_TRAINING = True
MODEL_FILENAME = 'best_model_nextqa_gdino_nolora_ncod_hum.ckpt'
FEAT_DIM = 1024

class Config:
    # Inputs
    video_feature_root = VIDEO_FEATURE_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    train_csv = TRAIN_CSV
    val_csv = VAL_CSV
    test_csv = TEST_CSV

    # Sampling sizes
    topK_frame = 16
    objs = 20
    frames = 16
    select_frames = 5
    topK_obj = 12

    frame_feat_dim = FEAT_DIM
    obj_feat_dim = GDINO_DIM
    use_grounding_dino = True

    # Model (align with Train_full_mode)
    d_model = 768
    word_dim = 768
    nheads = 8
    num_encoder_layers = 2
    num_decoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3

    # Text encoder (full fine-tuning, no LoRA)
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 1e-5
    text_pool_mode = 1
    use_lora = False

    # Training (Kaggle grad accumulation: 8 x 4 = 32)
    bs = 8
    accumulation_steps = 4
    lr = 1e-5
    epoch = 10
    gpu = 0
    gamma = 0.5
    decay = 1e-4
    n_query = 5
    lambda_verifier = 0.3
    lambda_knowledge = 0.2
    return_family_id = True

    # LR Scheduler + Early stopping
    lr_patience = 1
    min_lr = 1e-7
    early_stop_patience = 4
    early_stop_min_delta = 0.05
    early_stop_start_epoch = 5

    # NCOD + HUM
    ncod_u_lr = 0.1
    hum_alpha = 1.0

    # Other
    num_workers = 4
    hard_eval = False

args = Config()

set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Config: frame_feat_dim={args.frame_feat_dim}, obj_feat_dim={args.obj_feat_dim}')
print(f'Using GroundingDINO features: {args.use_grounding_dino}')
print(f'Gradient Accumulation: physical_bs={args.bs} x {args.accumulation_steps} = effective_bs={args.bs * args.accumulation_steps}')
print(f'Text encoder: {args.text_encoder_type} | use_lora={args.use_lora} | freeze_text_encoder={args.freeze_text_encoder}')

In [ ]:
# CELL 6: Datasets + Loaders (dùng DataLoaderNextQA)
print('=== CELL 6: Datasets ===')

print('Scanning features...')
VID_PT_MAP = scan_video_pt(args.video_feature_root)
GDINO_MAP = scan_gdino(args.grounding_dino_path)
print(f'  ViT .pt found: {len(VID_PT_MAP)}')
print(f'  GDINO .pkl found: {len(GDINO_MAP)}')

train_ds = NextQADataset(
    args.train_csv, VID_PT_MAP, GDINO_MAP,
    n_query=args.n_query, obj_num=args.objs, topK_frame=args.topK_frame,
    return_family_id=args.return_family_id, split='train'
)
val_ds = NextQADataset(
    args.val_csv, VID_PT_MAP, GDINO_MAP,
    n_query=args.n_query, obj_num=args.objs, topK_frame=args.topK_frame,
    return_family_id=args.return_family_id, split='val'
)
test_ds = NextQADataset(
    args.test_csv, VID_PT_MAP, GDINO_MAP,
    n_query=args.n_query, obj_num=args.objs, topK_frame=args.topK_frame,
    return_family_id=args.return_family_id, split='test'
)

train_loader = DataLoader(train_ds, args.bs, shuffle=True, num_workers=args.num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=True)

train_sample_keys = train_ds.sample_list['qns_key'].tolist()
train_key_to_idx = {k: i for i, k in enumerate(train_sample_keys)}

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# CELL 7: Model + optimizers + NCOD U (align with Train_full_mode, no LoRA)
print('=== CELL 7: Model ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames
model = VideoQAmodel(**cfg)
model.to(device)

non_text_params = []
text_base_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if 'text_encoder' in name:
        text_base_params.append(p)
    else:
        non_text_params.append(p)

param_groups = []
if len(non_text_params) > 0:
    param_groups.append({'params': non_text_params, 'lr': args.lr, 'weight_decay': args.decay})
if len(text_base_params) > 0:
    param_groups.append({'params': text_base_params, 'lr': args.text_encoder_lr, 'weight_decay': args.decay})
if len(param_groups) == 0:
    raise RuntimeError('No trainable parameters found for optimizer_model.')

optimizer_model = torch.optim.AdamW(param_groups)
scheduler = ReduceLROnPlateau(
    optimizer_model,
    mode='max',
    factor=args.gamma,
    patience=args.lr_patience,
    threshold=args.early_stop_min_delta,
    threshold_mode='abs',
    min_lr=args.min_lr
)

U = torch.nn.Parameter(torch.full((len(train_ds),), 1e-8, dtype=torch.float32, device=device))
optimizer_u = torch.optim.SGD([U], lr=args.ncod_u_lr)
xe = nn.CrossEntropyLoss()
bce = nn.BCEWithLogitsLoss()

BEST_ARTIFACT_NAME = 'best-model-nextqa-gdino-nolora-ncod-hum'
LAST_ARTIFACT_NAME = 'last-checkpoint-nextqa-gdino-nolora-ncod-hum'
save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)
LATEST_CKPT_PATH = os.path.join(MODEL_DIR, 'latest_checkpoint_nextqa_gdino_nolora_ncod_hum.ckpt')
TRAIN_HISTORY_CSV_PATH = os.path.join(MODEL_DIR, 'train_history_nextqa.csv')

print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')
print(f'Text-encoder trainable params: {sum(p.numel() for p in text_base_params)/1e6:.3f}M')
print(f'U shape: {tuple(U.shape)}')
print(f'Artifacts: best={BEST_ARTIFACT_NAME} | latest={LAST_ARTIFACT_NAME}')

In [ ]:
# CELL 8: Init W&B + (optional) resume
print('=== CELL 8: W&B init ===')
start_epoch = 1
best_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
history_rows = []

RESUME_FROM_CHECKPOINT = False
LOCAL_RESUME_PATH = LATEST_CKPT_PATH

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, config={
    'dataset': 'nextqa', 'backbone': 'dinov3+groundingdino',
    'text_encoder': args.text_encoder_type, 'use_lora': args.use_lora,
    'effective_batch_size': args.bs * args.accumulation_steps,
    'lambda_verifier': args.lambda_verifier, 'lambda_knowledge': args.lambda_knowledge,
    'ncod_u_lr': args.ncod_u_lr, 'hum_alpha': args.hum_alpha,
}, reinit=True, mode='online' if WANDB_API_KEY else 'disabled')
if WANDB_API_KEY:
    wandb.watch(model, log='gradients', log_freq=100)
    print(f'W&B run: {run.url}')

if RESUME_FROM_CHECKPOINT and os.path.exists(LOCAL_RESUME_PATH):
    ckpt = torch.load(LOCAL_RESUME_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    if 'optimizer_model_state_dict' in ckpt: optimizer_model.load_state_dict(ckpt['optimizer_model_state_dict'])
    if 'optimizer_u_state_dict' in ckpt: optimizer_u.load_state_dict(ckpt['optimizer_u_state_dict'])
    if 'scheduler_state_dict' in ckpt: scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'U' in ckpt:
        with torch.no_grad():
            u_ck = ckpt['U'].to(device).float().view(-1)
            n = min(u_ck.numel(), U.numel()); U[:n].copy_(u_ck[:n])
    best_acc = float(ckpt.get('best_acc', 0.0))
    start_epoch = int(ckpt.get('epoch', 0)) + 1
    print(f'Resumed at epoch {start_epoch} | best_acc={best_acc:.2f}')

In [ ]:
# CELL 9: Training loop (giữ nguyên bản causal, chỉ đổi tên file ckpt)
print('=== CELL 9: Training ===')
if RUN_TRAINING:
    for ep in range(start_epoch, args.epoch + 1):
        print(f'\nEpoch {ep}/{args.epoch}')
        total_loss, l1, l2, vloss, kloss, train_acc = train_epoch_integrated(
            model, optimizer_model, optimizer_u, U, train_loader, xe, bce, device, ep,
            train_key_to_idx, accumulation_steps=args.accumulation_steps,
            hum_alpha=args.hum_alpha, lambda_verifier=args.lambda_verifier,
            lambda_knowledge=args.lambda_knowledge
        )
        val_acc = eval_epoch(model, val_loader, device)
        scheduler.step(val_acc)
        history['train_loss'].append(total_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        epoch_row = {'epoch': ep, 'train_total_loss': float(total_loss), 'train_l1': float(l1),
                     'train_l2': float(l2), 'train_verifier_loss': float(vloss),
                     'train_knowledge_loss': float(kloss), 'train_acc': float(train_acc),
                     'val_acc': float(val_acc),
                     'u_mean': float(U.detach().mean().item()),
                     'u_max': float(U.detach().max().item()),
                     'lr_main': float(optimizer_model.param_groups[0]['lr'])}
        history_rows.append(epoch_row)
        pd.DataFrame(history_rows).to_csv(TRAIN_HISTORY_CSV_PATH, index=False)
        wandb.log(epoch_row)

        is_new_best = val_acc > best_acc
        if is_new_best:
            best_acc = val_acc
            print(f'New best val_acc={best_acc:.2f}%')
        ckpt = {'epoch': ep, 'model_state_dict': model.state_dict(),
                'optimizer_model_state_dict': optimizer_model.state_dict(),
                'optimizer_u_state_dict': optimizer_u.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_acc': best_acc, 'history': history,
                'history_rows': history_rows, 'U': U.detach().cpu(),
                'train_sample_keys': train_sample_keys}
        torch.save(ckpt, LATEST_CKPT_PATH)
        if is_new_best:
            torch.save(ckpt, save_path)
    print(f'Best Val Accuracy: {best_acc:.2f}%')
else:
    print('Skipping training')

In [ ]:
# CELL 10: NextQA evaluation — dump JSON {qid: {answer, prediction}} + accuracy_metric
print('=== CELL 10: NextQA Evaluation ===')


def predict_to_json(model, loader, device):
    model.eval()
    preds = {}
    with torch.no_grad():
        for batch in tqdm(loader, desc='Predict'):
            ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of = ff.to(device), of.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=q_family_id)
            logits = out.get('fused_score', out['logits'])
            pred = logits.argmax(-1).cpu().numpy()
            tgt = ans_id.numpy()
            for k, p, t in zip(qns_keys, pred, tgt):
                preds[str(k)] = {'answer': int(t), 'prediction': int(p)}
    return preds


val_preds = predict_to_json(model, val_loader, device)
test_preds = predict_to_json(model, test_loader, device)

VAL_PRED_JSON = os.path.join(MODEL_DIR, 'nextqa_val_predictions.json')
TEST_PRED_JSON = os.path.join(MODEL_DIR, 'nextqa_test_predictions.json')
with open(VAL_PRED_JSON, 'w') as f:
    json.dump(val_preds, f)
with open(TEST_PRED_JSON, 'w') as f:
    json.dump(test_preds, f)
print(f'Saved {VAL_PRED_JSON} / {TEST_PRED_JSON}')

print('\n--- VAL ---')
val_metrics, _, _ = accuracy_metric_nextqa(val_ds.sample_list, val_preds)
print('\n--- TEST ---')
test_metrics, _, _ = accuracy_metric_nextqa(test_ds.sample_list, test_preds)

final_metrics = {'val': val_metrics, 'test': test_metrics}
METRICS_JSON_PATH = os.path.join(MODEL_DIR, 'final_metrics_nextqa.json')
with open(METRICS_JSON_PATH, 'w') as f:
    json.dump(final_metrics, f, indent=2)
print(f'\nSaved {METRICS_JSON_PATH}')

if WANDB_API_KEY:
    wandb.log({f'val/{k}': v for k, v in val_metrics.items()})
    wandb.log({f'test/{k}': v for k, v in test_metrics.items()})

In [ ]:
# CELL 11: Finish W&B + upload artifacts
print('=== CELL 11: Finish W&B ===')

if WANDB_API_KEY:
    art = wandb.Artifact(
        name='final-results-nextqa-gdino-lora-ncod-hum',
        type='results',
        metadata={'dataset': 'nextqa'}
    )

    artifact_candidates = [
        'METRICS_JSON_PATH', 'VAL_PRED_JSON', 'TEST_PRED_JSON',
        'TRAIN_HISTORY_CSV_PATH', 'LATEST_CKPT_PATH', 'save_path'
    ]
    for var_name in artifact_candidates:
        p = globals().get(var_name, None)
        if p and os.path.exists(p):
            art.add_file(p, name=os.path.basename(p))

    wandb.log_artifact(art)
    wandb.finish()

print('Done.')